# Arabic Audio Transcriber — Local Notebook

Transcribes Arabic (and mixed Arabic/English) audio using **Faster-Whisper** on your local machine.

**Workflow:**
1. **Cell 3 — Batch transcribe** a folder of audio/video files (MP3, MP4, WAV, FLAC, M4A, MKV …). All timestamped transcripts (`[MM:SS-MM:SS]` lines) are saved directly to `transcript/<name>.txt` with no individual subfolders.
2. **Translate** the transcripts externally (prompt below), then put the translated `<name>.txt` files in the transcript folder — they are matched to media by name (`<name>.mp3`, `<name>.mp4`, …).
3. **Cell 8 — Batch final videos**: converts every `<name>.txt` into `<name>.srt`, then renders `Intro.mp4` + looping `Background.mp4` + the audio + burned-in subtitles → `output/<name>_final.mp4`.

**Also available:** Cell 7 burns an existing `.srt` into an existing video → `*_captioned.mp4`.

---
**Prerequisites (one-time setup):**
- Python ≥ 3.11 with a virtualenv activated
- `pip install faster-whisper yt-dlp torch tqdm`
- PyTorch CUDA build (if using GPU): `pip install torch --index-url https://download.pytorch.org/whl/cu128`
- [ffmpeg](https://ffmpeg.org/download.html) installed and on your PATH

Run **Cell 1** to verify all dependencies are present, then work top to bottom.

# Prompt

Translate the attached transcript into English. Preserve the intended meaning and word choice as closely as possible.
This is a generated transcript, so it may contain minor errors. Use the surrounding context to correct obvious transcription errors, but do not make large interpretive changes. For context, this is Islamic material.

Format the output as a timestamped transcript, for example:
[00:00-00:05] Hello everyone.

Timestamp rules:
Keep the original timestamp blocks and their order. Do not invent new timestamps, remove timestamps, extend timestamps, shorten timestamps, or change the start/end time of any timestamp block.
The translated transcript must begin and end at exactly the same timestamps as the source chunk.
You may only adjust caption balance by moving translated words between adjacent existing timestamp blocks when the speech clearly continues across them. Do not create new time ranges.
If a source line is very short, keep it short unless moving nearby words into it is necessary for readability.
If a source line is very long, keep the same timestamp and translate it fully; do not split it into new timestamps.

Continuity:
This video is long, so I will provide it in 20-minute chunks. Treat each part as continuing from the previous part, not as an independent transcript. Preserve terminology consistently across all parts.


Output rules:
Do not include notes, explanations, headings, corrections, or commentary.
Do not include anything other than the translated transcript.
Preserve the original wording as much as possible.

REMINDER: PRESERVE ORIGINAL WORDING AND ORIGINAL TIMESTAMP BOUNDARIES.

In [1]:
# ── Cell 1: Verify dependencies ──────────────────────────────────────────────
import importlib, shutil, sys

missing = [pkg for pkg in ["faster_whisper", "yt_dlp", "torch", "tqdm"] if importlib.util.find_spec(pkg) is None]
if missing:
    print("Missing Python packages:", ", ".join(missing))
    print("Install with:  pip install", " ".join(p.replace('_', '-') for p in missing))
else:
    print("✓ Python packages OK")

if importlib.util.find_spec("hf_xet") is None:
    print("⚠ 'hf_xet' not installed — model downloads from Hugging Face will be slow.")
    print("  Fix:  pip install hf_xet   (then Kernel → Restart)")
else:
    print("✓ hf_xet OK — fast Hugging Face downloads enabled")

if shutil.which("ffmpeg"):
    import subprocess
    v = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0]
    print(f"✓ ffmpeg: {v}")
else:
    print("✗ ffmpeg not found — install from https://ffmpeg.org/download.html and add it to PATH")

import torch
print(f"Python : {sys.executable}")
print(f"torch  : {torch.__version__}")

if torch.cuda.is_available():
    print(f"✓ GPU  : {torch.cuda.get_device_name(0)}")
elif "+cpu" in torch.__version__:
    print("⚠ CPU-only PyTorch (+cpu) — transcription will use CPU.")
    print("  Fix (in the project .venv, with it activated):")
    print("    pip uninstall torch torchaudio -y")
    print("    pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu126")
    print("  Then: Kernel → Restart, and re-run this cell.")
else:
    print("⚠ torch.cuda.is_available() is False — will use CPU.")
    print("  Run:  python check_cuda.py   from the project folder for details.")

✓ Python packages OK
✓ hf_xet OK — fast Hugging Face downloads enabled
✓ ffmpeg: ffmpeg version 4.3 Copyright (c) 2000-2020 the FFmpeg developers
Python : c:\Users\subha\anaconda3\envs\transcribe\python.exe
torch  : 2.5.1
✓ GPU  : NVIDIA GeForce RTX 3070


## ⚙️ Configuration

Edit the values in the cell below before running anything else.

| Setting | Options | Notes |
|---------|---------|-------|
| `MODEL_SIZE` | `tiny` `base` `small` `medium` `large-v2` | Larger = more accurate, slower |
| `LANGUAGE` | `"ar"`, `"en"`, `"fr"` … or `None` | `None` = auto-detect from first 30 s |
| `START_TIME` | `"8:00"`, `"1:25:00"`, `None` | `None` = from beginning (applies to **every** file in the batch) |
| `END_TIME` | `"25:00"`, `None` | `None` = to end of file (applies to **every** file in the batch) |

In [2]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
import re
import torch
from pathlib import Path

MODEL_SIZE = "large-v3-turbo"   # tiny | base | small | medium | large-v2
LANGUAGE   = "ar"          # e.g. "ar", "en", "fr", or None to auto-detect

# Optional time range — set to None to transcribe the whole file
START_TIME = None          # e.g. "8:00" or "1:25:00"
END_TIME   = None          # e.g. "25:00"

# YouTube video download: max height in pixels (720, 1080, …). Used in Cell 3B.
MAX_VIDEO_HEIGHT = 720

# Transcripts from Cell 3 are saved here (created automatically)
TRANSCRIPTS_DIR = Path("transcript")

# ── Helpers ───────────────────────────────────────────────────────────────────
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

def _parse_ts(s):
    """Parse 'SS', 'M:SS', 'MM:SS', or 'H:MM:SS' into seconds (float)."""
    if s is None:
        return None
    parts = str(s).strip().split(":")
    nums  = [float(p) for p in parts]
    if len(nums) == 1: return nums[0]
    if len(nums) == 2: return nums[0] * 60 + nums[1]
    return nums[0] * 3600 + nums[1] * 60 + nums[2]

def _fmt_ts(t):
    """Format seconds as MM:SS or H:MM:SS."""
    s = int(round(t))
    h, r = divmod(s, 3600)
    m, sec = divmod(r, 60)
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m:02d}:{sec:02d}"

def _slugify(text: str, max_len: int = 60) -> str:
    """Filesystem-safe name (keeps letters/digits/Arabic, collapses spaces to _)."""
    text = text.strip().lower()
    text = re.sub(r"[^\w\s-]", "", text, flags=re.UNICODE)
    text = re.sub(r"[\s_-]+", "_", text)
    return text.strip("_")[:max_len] or "output"

def _unique_dir(base: Path) -> Path:
    """Return base if unused, otherwise base_2, base_3, …"""
    if not base.exists():
        return base
    i = 2
    while True:
        candidate = base.parent / f"{base.name}_{i}"
        if not candidate.exists():
            return candidate
        i += 1

start_sec = _parse_ts(START_TIME)
end_sec   = _parse_ts(END_TIME)

print(f"Model   : {MODEL_SIZE}  |  Device: {DEVICE}  |  Compute: {COMPUTE_TYPE}")
print(f"Language: {LANGUAGE or 'auto-detect'}")
if start_sec is not None or end_sec is not None:
    s = _fmt_ts(start_sec or 0)
    e = _fmt_ts(end_sec) if end_sec is not None else "end"
    print(f"Range   : {s} → {e}")
else:
    print("Range   : full file")

Model   : large-v3-turbo  |  Device: cuda  |  Compute: float16
Language: ar
Range   : full file


## 📦 Model cache

Downloads `MODEL_SIZE` from Hugging Face **once** and reuses the local cache on every later run — no re-download, no network call, unless you explicitly ask to check for updates.

In [3]:
# ── Cell 2B: Model cache — download once, reuse offline ──────────────────────
#
# First run of a new MODEL_SIZE: downloads the model (this is the slow part —
# install `hf_xet` per Cell 1 to speed it up).
# Every later run: loads straight from the local Hugging Face cache with
# local_files_only=True, so there is zero network traffic and zero delay.
#
# Set CHECK_FOR_MODEL_UPDATES = True and re-run this cell to check Hugging Face
# for a newer revision of the model (only re-downloads if something changed).

from pathlib import Path
from faster_whisper.utils import download_model

CHECK_FOR_MODEL_UPDATES = False


def _is_complete(model_dir) -> bool:
    """local_files_only=True happily returns a snapshot dir even if the big
    weights file never finished downloading (e.g. a previous run was
    interrupted). Verify model.bin actually resolves before trusting it."""
    p = Path(model_dir) / "model.bin"
    return p.exists() and p.stat().st_size > 0


MODEL_PATH = None
if not CHECK_FOR_MODEL_UPDATES:
    try:
        candidate = download_model(MODEL_SIZE, local_files_only=True)
        if _is_complete(candidate):
            MODEL_PATH = candidate
            print(f"✓ Model '{MODEL_SIZE}' ready (local cache): {MODEL_PATH}")
        else:
            print(f"'{MODEL_SIZE}' is cached but incomplete (missing/empty model.bin) — will re-download.")
    except Exception:
        pass

if MODEL_PATH is None:
    print(f"Downloading '{MODEL_SIZE}' (one-time — this is the slow part; hf_xet speeds it up)…")
    MODEL_PATH = download_model(MODEL_SIZE, local_files_only=False)
    if not _is_complete(MODEL_PATH):
        raise RuntimeError(f"Download finished but model.bin is still missing/empty in {MODEL_PATH}")
    print(f"✓ Downloaded and cached: {MODEL_PATH}")

C:\Users\subha\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Model 'large-v3-turbo' ready (local cache): C:\Users\subha\.cache\huggingface\hub\models--mobiuslabsgmbh--faster-whisper-large-v3-turbo\snapshots\0a363e9161cbc7ed1431c9597a8ceaf0c4f78fcf


## 📥 Batch transcription

Set `INPUT_FOLDER` in the cell below to a folder of audio/video files. Every file in the folder is transcribed **in succession**, and all timestamped transcripts are saved directly in one folder:

```
transcript/<name>.txt
```

No individual subfolders are created. Transcript filenames match the media filenames exactly (`test.mp3` → `transcript/test.txt`), so after translating them externally (prompt at the top of this notebook) you can drop the translated `.txt` files back into the transcript folder and run the final-video cell at the bottom.

In [4]:
# ── Cell 3: Batch transcribe a folder ─────────────────────────────────────────
#
# Transcribes every audio/video file in INPUT_FOLDER, one after another, and
# saves all timestamped transcripts directly in TRANSCRIPTS_DIR:
#   transcript/<name>.txt        (test.mp3 → transcript/test.txt)
#
# Files that already have a transcript are skipped — set OVERWRITE = True to
# re-transcribe everything.

from faster_whisper import WhisperModel

INPUT_FOLDER = r"input"   # ← folder containing the audio/video files
OVERWRITE    = False      # True = re-transcribe files that already have a transcript

MEDIA_EXTS = {
    ".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg", ".opus", ".wma",
    ".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v",
}

in_dir = Path(INPUT_FOLDER)
if not in_dir.is_dir():
    raise FileNotFoundError(f"Folder not found: {in_dir.resolve()}")

media_files = sorted(
    f for f in in_dir.iterdir()
    if f.is_file() and f.suffix.lower() in MEDIA_EXTS
)
if not media_files:
    raise FileNotFoundError(f"No audio/video files found in: {in_dir.resolve()}")

TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Folder      : {in_dir.resolve()}")
print(f"Transcripts : {TRANSCRIPTS_DIR.resolve()}")
print(f"Files       : {len(media_files)}")
for f in media_files:
    print(f"  • {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

# ── Load model once for the whole batch ───────────────────────────────────────
print(f"\nLoading Faster-Whisper '{MODEL_SIZE}' on {DEVICE} ({COMPUTE_TYPE}) …")
model = WhisperModel(MODEL_PATH, device=DEVICE, compute_type=COMPUTE_TYPE)

# Optional time range from Cell 2 (applies to every file in the batch)
use_clip   = start_sec is not None or end_sec is not None
clip_start = start_sec or 0.0
if not use_clip:
    clip_timestamps = "0"
    vad_filter = True
elif end_sec is None:
    clip_timestamps = str(clip_start)
    vad_filter = False
else:
    clip_timestamps = f"{clip_start},{float(end_sec)}"
    vad_filter = False

# ── Transcribe each file in succession ────────────────────────────────────────
done, skipped, failed = [], [], []

for idx, media in enumerate(media_files, 1):
    out_txt = TRANSCRIPTS_DIR / f"{media.stem}.txt"
    print(f"\n{'─' * 70}\n[{idx}/{len(media_files)}] {media.name}")

    if out_txt.exists() and not OVERWRITE:
        print(f"  ↷ Skipped — transcript already exists: {out_txt}")
        skipped.append(media.name)
        continue

    try:
        segments, info = model.transcribe(
            str(media),
            language=LANGUAGE,
            beam_size=5,
            temperature=0.0,
            condition_on_previous_text=False,
            vad_filter=vad_filter,
            clip_timestamps=clip_timestamps,

            # Helpful for long-form transcription
            word_timestamps=False,
            compression_ratio_threshold=2.4,
            log_prob_threshold=-1.0,
            no_speech_threshold=0.6,
        )

        duration = getattr(info, "duration", None)
        if duration:
            print(f"  Audio length: {_fmt_ts(duration)}")

        timed_lines = []
        for seg in segments:
            text = seg.text.strip()
            if not text:
                continue
            line = f"[{_fmt_ts(float(seg.start or 0))}-{_fmt_ts(float(seg.end or 0))}] {text}"
            timed_lines.append(line)
            print(f"  {line}")

        out_txt.write_text("\n".join(timed_lines), encoding="utf-8")
        print(f"  ✓ Saved {len(timed_lines)} segments → {out_txt}")
        done.append(media.name)

    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        failed.append(media.name)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═' * 70}")
print(f"✓ Transcribed: {len(done)}   ↷ Skipped: {len(skipped)}   ✗ Failed: {len(failed)}")
for name in failed:
    print(f"  ✗ {name}")
print(f"Transcripts saved in: {TRANSCRIPTS_DIR.resolve()}")

Folder      : C:\Users\subha\Desktop\transcribe\youtube-translator\input
Transcripts : C:\Users\subha\Desktop\transcribe\youtube-translator\transcript
Files       : 1
  • truth.mp3  (2.1 MB)

Loading Faster-Whisper 'large-v3-turbo' on cuda (float16) …

──────────────────────────────────────────────────────────────────────
[1/1] truth.mp3
  Audio length: 01:29
  [00:02-00:06] حقيقة دعوة شيخ الإسلام محمد بن عبد الوهاب
  [00:06-00:10] أصل دعوة شيخ الإسلام محمد بن عبد الوهاب إلى التوحيد
  [00:10-00:13] إلى إفراد الله بالعبادة
  [00:13-00:18] ولم يكن في العارض ذاك الوقت أحد من أهل العلم يعرف التوحيد
  [00:18-00:24] حتى إن والد شيخ الإسلام وجده ما كان يعرفون التوحيد
  [00:24-00:28] وإنما هداه الله إلى معرفة التوحيد وإفراد الله بالعبادة
  [00:28-00:33] عن طريق قراءة كتب الحديث والتفسير
  [00:33-00:36] ذكر هذا العلام عبد الحمد الحسن في كتابه المقامات
  [00:36-00:41] وكان مريدا أن يصدع به لكن أباه كان يمنعه
  [00:41-00:45] فلما توفي أبوه صدعا صدعا بهذه الدعوة
  [00:45-00:47] وعارضه علماء زمانه


## 🎞️ Batch final videos from translated transcripts

**Cell 8** processes a whole batch in succession. Point it at **two folders**:

- `MEDIA_FOLDER` — the **original media** files — `<name>.mp3`, `<name>.mp4`, …
- `TRANSCRIPT_FOLDER` — the **translated** timestamped transcripts — `<name>.txt` with `[MM:SS-MM:SS]` lines

Files are matched across the two folders by shared name (`test.txt` in `TRANSCRIPT_FOLDER` pairs with `test.mp3` in `MEDIA_FOLDER`).

For each pair it:

1. Converts `<name>.txt` → `<name>.srt` (no separate SRT cell needed).
2. Builds the final video:
   - `Intro.mp4` plays first (with its own audio).
   - `Background.mp4` loops under the main audio until it ends — the last loop plays out fully instead of being cut to the audio.
   - Captions are automatically shifted by the intro duration so they stay in sync.
   - The clip ends with a fade to black.
3. Saves `<name>_final.mp4` to `OUTPUT_FOLDER` (default `output/`).

In [5]:
# ── Cell 8: Batch final videos — txt → SRT → intro + background + subtitles ──
#
# Point MEDIA_FOLDER at the original media (<name>.mp3 / <name>.mp4 / …) and
# TRANSCRIPT_FOLDER at the translated transcripts (<name>.txt, [MM:SS-MM:SS]
# lines). Pairs are matched across the two folders by name: test.txt ↔ test.mp3.
#
# For each pair, in succession:
#   1. <name>.txt → <name>.srt   (saved next to the transcript)
#   2. Intro.mp4 → Background.mp4 looped under the audio → fade to black,
#      with the SRT shifted by the intro duration → OUTPUT_FOLDER/<name>_final.mp4
#
# Encoding prefers the GPU (NVENC): HEVC (h265) first for the best
# compression, then H.264, falling back to CPU libx264 if no NVENC is present.

import re
import shutil
import subprocess
from math import ceil
from pathlib import Path

MEDIA_FOLDER      = r"input"          # folder of original media files (<name>.mp3 / .mp4 / …)
TRANSCRIPT_FOLDER = r"transcript"     # folder of translated transcripts (<name>.txt)
OUTPUT_FOLDER     = r"output"         # final videos are saved here (created automatically)
INTRO_PATH        = "assets/Intro.mp4"       # plays first, with its own audio
BACKGROUND_PATH   = "assets/Background.mp4"  # loops under the main audio
OVERWRITE_FINAL   = False             # True = re-render videos that already exist

FADE_SECONDS = 1.5     # fade-to-black duration at the very end
TARGET_W, TARGET_H, TARGET_FPS = 1920, 1080, 24

# Encoder selection
PREFER_GPU  = True     # True = use NVENC (GPU) when available — much faster
PREFER_HEVC = True     # True = prefer h265 (HEVC) over h264 on NVENC — smaller files
FFMPEG_PATH = None     # e.g. r"C:\ffmpeg\bin\ffmpeg.exe"; None = auto-detect a full build

# Encode quality: used as CRF for libx264 (lower = better) and as CQ for NVENC.
VIDEO_CRF     = 18
VIDEO_PRESET  = "fast"   # libx264 preset (ultrafast … veryslow)
NVENC_PRESET  = "p5"       # NVENC preset p1 (fastest) … p7 (best quality)
AUDIO_BITRATE = "192k"

SUBTITLE_STYLE = (
    "FontName=Arial,"
    "FontSize=16,"
    "PrimaryColour=&H00FFFFFF,"
    "BackColour=&H00000000,"
    "BorderStyle=3,"
    "MarginV=40,"
    "Alignment=2"
)

# ── Resolve folder, intro, background ─────────────────────────────────────────
_nb_dir = Path.cwd()

def _final_resolve(p: str | Path, exts: set[str], label: str) -> Path:
    p = Path(str(p).strip().strip('"').strip("'")).expanduser()
    if not p.is_absolute():
        p = _nb_dir / p
    if not p.is_file():
        raise FileNotFoundError(f"{label} not found: {p}")
    if p.suffix.lower() not in exts:
        raise ValueError(f"{label}: expected {exts}, got {p.suffix!r}: {p}")
    return p.resolve()

_MEDIA_EXTS = {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v", ".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg", ".opus"}

def _resolve_dir(p: str | Path, label: str) -> Path:
    p = Path(str(p).strip().strip('"').strip("'")).expanduser()
    if not p.is_absolute():
        p = _nb_dir / p
    if not p.is_dir():
        raise FileNotFoundError(f"{label} not found: {p}")
    return p.resolve()

media_dir      = _resolve_dir(MEDIA_FOLDER, "MEDIA_FOLDER")
transcript_dir = _resolve_dir(TRANSCRIPT_FOLDER, "TRANSCRIPT_FOLDER")

output_dir = Path(str(OUTPUT_FOLDER).strip().strip('"').strip("'")).expanduser()
if not output_dir.is_absolute():
    output_dir = _nb_dir / output_dir
output_dir.mkdir(parents=True, exist_ok=True)
output_dir = output_dir.resolve()

_intro = _final_resolve(INTRO_PATH, {".mp4", ".mkv", ".mov", ".webm", ".m4v"}, "INTRO_PATH")
_bg    = _final_resolve(BACKGROUND_PATH, {".mp4", ".mkv", ".mov", ".webm", ".m4v"}, "BACKGROUND_PATH")

# ── Pair transcripts (TRANSCRIPT_FOLDER) with media (MEDIA_FOLDER) by name ────
txt_files = sorted(f for f in transcript_dir.iterdir() if f.is_file() and f.suffix.lower() == ".txt")
if not txt_files:
    raise FileNotFoundError(f"No .txt transcripts found in: {transcript_dir}")

pairs, unmatched = [], []
for txt in txt_files:
    media = [
        f for f in media_dir.iterdir()
        if f.is_file() and f.stem == txt.stem and f.suffix.lower() in _MEDIA_EXTS
        and not f.stem.endswith("_final")
    ]
    if media:
        pairs.append((txt, media[0]))
    else:
        unmatched.append(txt.name)

print(f"Media folder      : {media_dir}")
print(f"Transcript folder : {transcript_dir}")
print(f"Output folder     : {output_dir}")
print(f"Pairs  : {len(pairs)}")
for txt, media in pairs:
    print(f"  • {txt.name}  ↔  {media.name}")
for name in unmatched:
    print(f"  ⚠ {name} — no matching media file, skipping")
if not pairs:
    raise FileNotFoundError("No transcript/media pairs found (they must share the same name).")

# ── Resolve ffmpeg / ffprobe ──────────────────────────────────────────────────
def _ffmpeg_candidates() -> list[Path]:
    seen, candidates = set(), []

    def _add(path):
        if path is None:
            return
        key = str(path.resolve()).lower()
        if key not in seen:
            seen.add(key)
            candidates.append(path)

    if FFMPEG_PATH:
        _add(Path(str(FFMPEG_PATH).strip().strip('"').strip("'")).expanduser())

    # Prefer known full builds over PATH (a kernel can keep a stale PATH entry).
    for pattern in (r"C:\ffmpeg\bin\ffmpeg.exe", r"C:\Program Files\ffmpeg\bin\ffmpeg.exe"):
        _add(Path(pattern))

    winget_root = Path.home() / "AppData/Local/Microsoft/WinGet/Packages"
    if winget_root.is_dir():
        for exe in sorted(winget_root.glob("Gyan.FFmpeg*/ffmpeg-*/bin/ffmpeg.exe"), reverse=True):
            _add(exe)

    which = shutil.which("ffmpeg")
    if which:
        _add(Path(which))
    return candidates


def _available_encoders(ffmpeg: Path) -> set[str]:
    """Return exact encoder names from `ffmpeg -encoders`."""
    probe = subprocess.run([str(ffmpeg), "-hide_banner", "-encoders"], capture_output=True, text=True)
    if probe.returncode != 0:
        return set()
    names = set()
    for line in probe.stdout.splitlines():
        parts = line.split()
        # Encoder rows are: six capability flags, encoder name, description.
        if len(parts) >= 2 and len(parts[0]) == 6 and parts[0][0] in "VAS":
            names.add(parts[1])
    return names


_USABLE_VIDEO_ENCODERS = {"libx264", "h264_nvenc", "hevc_nvenc", "h264_mf"}


def _resolve_ffmpeg() -> Path:
    for exe in _ffmpeg_candidates():
        if not exe.is_file():
            continue
        if not (_USABLE_VIDEO_ENCODERS & _available_encoders(exe)):
            continue
        version_lines = subprocess.run([str(exe), "-version"], capture_output=True, text=True).stdout.splitlines()
        print(f"ffmpeg : {exe}")
        if version_lines:
            print(f"         {version_lines[0]}")
        return exe
    raise RuntimeError(
        "No ffmpeg with a usable H.264/HEVC encoder found.\n"
        "Install a full build (e.g. winget install Gyan.FFmpeg) or set FFMPEG_PATH."
    )


def _video_encode_args(ffmpeg: Path, cq: int, x264_preset: str) -> tuple[str, list[str]]:
    """Pick the fastest/best encoder. Prefers NVENC (GPU); HEVC before H.264
    when PREFER_HEVC; falls back to CPU libx264, then h264_mf."""
    encoders = _available_encoders(ffmpeg)

    if PREFER_GPU and PREFER_HEVC and "hevc_nvenc" in encoders:
        # -tag:v hvc1 keeps the HEVC MP4 playable in QuickTime/Apple players.
        return "hevc_nvenc", ["-c:v", "hevc_nvenc", "-preset", NVENC_PRESET,
                              "-rc", "vbr", "-cq", str(cq), "-b:v", "0", "-tag:v", "hvc1"]
    if PREFER_GPU and "h264_nvenc" in encoders:
        return "h264_nvenc", ["-c:v", "h264_nvenc", "-preset", NVENC_PRESET,
                              "-rc", "vbr", "-cq", str(cq), "-b:v", "0"]
    if PREFER_GPU and "hevc_nvenc" in encoders:  # HEVC available but PREFER_HEVC off and no h264_nvenc
        return "hevc_nvenc", ["-c:v", "hevc_nvenc", "-preset", NVENC_PRESET,
                              "-rc", "vbr", "-cq", str(cq), "-b:v", "0", "-tag:v", "hvc1"]
    if "libx264" in encoders:
        return "libx264", ["-c:v", "libx264", "-crf", str(cq), "-preset", x264_preset]
    if "h264_mf" in encoders:
        quality = max(10, min(100, 110 - cq * 3))
        return "h264_mf", ["-c:v", "h264_mf", "-rate_control", "quality", "-quality", str(quality)]
    raise RuntimeError("ffmpeg has no usable H.264/HEVC encoder (libx264, h264_nvenc, hevc_nvenc, h264_mf).")


_ffmpeg = _resolve_ffmpeg()
_encoder, _encode_args = _video_encode_args(_ffmpeg, VIDEO_CRF, VIDEO_PRESET)
print(f"Encoder : {_encoder}")

_ffprobe = _ffmpeg.with_name("ffprobe.exe" if _ffmpeg.suffix else "ffprobe")
if not _ffprobe.is_file():
    _ffprobe = Path(shutil.which("ffprobe") or "")
    if not _ffprobe.is_file():
        raise RuntimeError("ffprobe not found next to ffmpeg or on PATH.")

def _duration(path: Path) -> float:
    r = subprocess.run(
        [str(_ffprobe), "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(path)],
        capture_output=True, text=True,
    )
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError(f"ffprobe failed for {path}:\n{r.stderr[-1000:]}")
    return float(r.stdout.strip())

intro_dur = _duration(_intro)
bg_dur    = _duration(_bg)
print(f"Intro      : {_intro.name}  ({intro_dur:.2f}s)")
print(f"Background : {_bg.name}  ({bg_dur:.2f}s per loop)")

# ── Transcript → SRT ──────────────────────────────────────────────────────────
_ts_line = re.compile(r"\[(\d+(?::\d{2})+)\s*-\s*(\d+(?::\d{2})+)\]\s*(.*)")

def _ts_to_seconds(ts):
    nums = [int(p) for p in ts.split(":")]
    if len(nums) == 2: return nums[0] * 60 + nums[1]
    if len(nums) == 3: return nums[0] * 3600 + nums[1] * 60 + nums[2]
    raise ValueError(f"Cannot parse: {ts!r}")

def _seconds_to_srt(t):
    ms = int(round(t * 1000))
    h,  rem = divmod(ms, 3_600_000)
    m,  rem = divmod(rem,    60_000)
    s,  ms  = divmod(rem,     1_000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def _txt_to_srt(txt_path: Path) -> Path:
    entries, counter = [], 1
    for line in txt_path.read_text(encoding="utf-8").splitlines():
        mat = _ts_line.match(line.strip())
        if not mat:
            continue
        start_raw, end_raw, text = mat.groups()
        text = text.strip()
        if not text:
            continue
        entries.append(
            f"{counter}\n{_seconds_to_srt(_ts_to_seconds(start_raw))} --> "
            f"{_seconds_to_srt(_ts_to_seconds(end_raw))}\n{text}\n"
        )
        counter += 1
    if not entries:
        raise RuntimeError(f"No [MM:SS-MM:SS] lines found in {txt_path.name}. Check the transcript format.")
    srt_path = txt_path.with_suffix(".srt")
    srt_path.write_text("\n".join(entries), encoding="utf-8")
    print(f"  ✓ {srt_path.name}  ({counter - 1} captions)")
    return srt_path

# ── Shift SRT by the intro duration ───────────────────────────────────────────
_srt_time = re.compile(r"(\d{2,}):(\d{2}):(\d{2}),(\d{3})")

def _shift_srt_text(text: str, offset_sec: float) -> str:
    off_ms = int(round(offset_sec * 1000))
    def _repl(m):
        h, mnt, s, ms = (int(g) for g in m.groups())
        t = h * 3_600_000 + mnt * 60_000 + s * 1_000 + ms + off_ms
        h, rem = divmod(t, 3_600_000)
        mnt, rem = divmod(rem, 60_000)
        s, ms = divmod(rem, 1_000)
        return f"{h:02d}:{mnt:02d}:{s:02d},{ms:03d}"
    return _srt_time.sub(_repl, text)

# ── Render one pair ───────────────────────────────────────────────────────────
def _render_final(audio: Path, srt: Path, out_final: Path) -> Path:
    audio_dur = _duration(audio)

    # Last background loop plays out fully instead of being trimmed to the audio.
    bg_loops  = max(1, ceil(audio_dur / bg_dur))
    total_dur = intro_dur + bg_loops * bg_dur
    fade_dur  = min(FADE_SECONDS, total_dur)

    print(f"  Audio {audio_dur:.2f}s → {bg_loops} bg loops, total {total_dur:.2f}s (fade last {fade_dur:.1f}s)")

    # subtitles filter chokes on Windows absolute paths → work in the audio's
    # folder with a relative .srt name (the output path can be absolute).
    work_dir  = audio.parent
    shifted_srt = work_dir / "_final_subs.srt"
    shifted_srt.write_text(_shift_srt_text(srt.read_text(encoding="utf-8"), intro_dur), encoding="utf-8")

    _scale = (
        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=decrease,"
        f"pad={TARGET_W}:{TARGET_H}:-1:-1:color=black,setsar=1,fps={TARGET_FPS}"
    )
    filter_complex = (
        f"[0:v]{_scale}[iv];"
        f"[1:v]{_scale}[bv];"
        f"[iv][bv]concat=n=2:v=1:a=0[v0];"
        f"[v0]subtitles=_final_subs.srt:force_style='{SUBTITLE_STYLE}'[v1];"
        f"[v1]fade=t=out:st={total_dur - fade_dur:.3f}:d={fade_dur:.3f}[vout];"
        f"[0:a]aresample=48000,aformat=channel_layouts=stereo[ia];"
        f"[2:a]aresample=48000,aformat=channel_layouts=stereo[ma];"
        f"[ia][ma]concat=n=2:v=0:a=1[a0];"
        f"[a0]apad=whole_dur={total_dur:.3f}[aout]"
    )

    cmd = [
        str(_ffmpeg), "-y",
        "-i", str(_intro),
        "-stream_loop", str(bg_loops - 1), "-i", str(_bg),
        "-i", str(audio),
        "-filter_complex", filter_complex,
        "-map", "[vout]", "-map", "[aout]",
        *_encode_args,
        "-c:a", "aac", "-b:a", AUDIO_BITRATE,
        "-t", f"{total_dur:.3f}",
        "-movflags", "+faststart",
        str(out_final),
    ]

    print(f"  Encoding with {_encoder} → {out_final.name} …")
    try:
        result = subprocess.run(cmd, cwd=work_dir, capture_output=True, text=True)
        if result.returncode != 0:
            print("  ffmpeg failed:\n", result.stderr[-4000:])
            raise RuntimeError(f"ffmpeg exited with code {result.returncode}")
    finally:
        shifted_srt.unlink(missing_ok=True)

    print(f"  ✓ Wrote {out_final}  ({out_final.stat().st_size / 1e6:.1f} MB)")
    return out_final

# ── Process every pair in succession ──────────────────────────────────────────
done, skipped, failed = [], [], []

for idx, (txt, media) in enumerate(pairs, 1):
    print(f"\n{'─' * 70}\n[{idx}/{len(pairs)}] {media.name}")
    out_final = output_dir / f"{media.stem}_final.mp4"

    if out_final.exists() and not OVERWRITE_FINAL:
        print(f"  ↷ Skipped — already exists: {out_final.name}")
        skipped.append(media.name)
        continue

    try:
        srt = _txt_to_srt(txt)
        _render_final(media, srt, out_final)
        done.append(media.name)
    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        failed.append(media.name)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═' * 70}")
print(f"✓ Rendered: {len(done)}   ↷ Skipped: {len(skipped)}   ✗ Failed: {len(failed)}")
for name in failed:
    print(f"  ✗ {name}")
print(f"Final videos saved in: {output_dir}")

Media folder      : C:\Users\subha\Desktop\transcribe\youtube-translator\input
Transcript folder : C:\Users\subha\Desktop\transcribe\youtube-translator\transcript
Output folder     : C:\Users\subha\Desktop\transcribe\youtube-translator\output
Pairs  : 1
  • truth.txt  ↔  truth.mp3
ffmpeg : C:\ffmpeg\bin\ffmpeg.exe
         ffmpeg version 7.1.1-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
Encoder : hevc_nvenc
Intro      : Intro.mp4  (10.01s)
Background : Background.mp4  (8.17s per loop)

──────────────────────────────────────────────────────────────────────
[1/1] truth.mp3
  ✓ truth.srt  (21 captions)
  Audio 89.40s → 11 bg loops, total 99.88s (fade last 1.5s)
  Encoding with hevc_nvenc → truth_final.mp4 …
  ✓ Wrote C:\Users\subha\Desktop\transcribe\youtube-translator\output\truth_final.mp4  (64.5 MB)

══════════════════════════════════════════════════════════════════════
✓ Rendered: 1   ↷ Skipped: 0   ✗ Failed: 0
Final videos saved in: C:\Users\subha\Desktop\tr